In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore', category=UserWarning, module='openpyxl')

def transform_excel_to_csv(input_file, output_file):
    print(f"Processing: {input_file}")
    
    # 1. 헤더 구조 파악용 상위 행 읽기
    try:
        df_raw = pd.read_excel(input_file, header=None, nrows=10)
    except Exception as e:
        print(f"파일 읽기 실패: {e}")
        return

    # 2. '약효분류코드'가 포함된 행 찾기
    header_row_idx = None
    for i in range(len(df_raw)):
        row_values = df_raw.iloc[i].values
        if any('약효분류코드' in str(val) for val in row_values):
            header_row_idx = i
            break
    
    if header_row_idx is None:
        print("오류: '약효분류코드'를 포함하는 행을 찾을 수 없습니다.")
        return
    
    print(f"  → 헤더 행 위치: {header_row_idx}번째 행")

    # 3. 상단 헤더(월 정보) 존재 여부 확인
    has_top_header = False
    if header_row_idx > 0:
        top_row_values = df_raw.iloc[header_row_idx - 1].values
        if any('년' in str(val) and '월' in str(val) for val in top_row_values):
            has_top_header = True

    # 4. 엑셀 다시 읽기
    if has_top_header:
        df = pd.read_excel(input_file, header=[header_row_idx - 1, header_row_idx])
    else:
        df = pd.read_excel(input_file, header=header_row_idx)

    # 5. 컬럼명 평탄화
    if isinstance(df.columns, pd.MultiIndex):
        new_columns = []
        for col in df.columns:
            clean_parts = [str(c).strip() for c in col if 'Unnamed' not in str(c) and str(c) != 'nan']
            new_columns.append('_'.join(clean_parts) if clean_parts else '')
        df.columns = new_columns
    else:
        df.columns = [str(c).strip() for c in df.columns]

    # 6. 컬럼명 표준화
    rename_dict = {}
    for col in df.columns:
        if '약효분류코드' in col:
            rename_dict[col] = '약효분류코드'
        elif '약효분류명' in col:
            rename_dict[col] = '약효분류명'
    df.rename(columns=rename_dict, inplace=True)

    if '약효분류코드' not in df.columns or '약효분류명' not in df.columns:
        print("오류: '약효분류코드' 또는 '약효분류명' 컬럼을 찾을 수 없습니다.")
        print("현재 컬럼 목록:", df.columns.tolist())
        return

    # 7. '계' 행 제거
    df = df[df['약효분류명'] != '계']

    # 8. Melt
    id_vars = ['약효분류코드', '약효분류명']
    value_vars = [col for col in df.columns if col not in id_vars]
    
    df_melted = df.melt(id_vars=id_vars, value_vars=value_vars, 
                        var_name='월_구분', value_name='값')

    # 9. 월_구분 분리
    split_cols = df_melted['월_구분'].str.split('_', n=1, expand=True)
    df_melted['진료년월_raw'] = split_cols[0]
    df_melted['구분'] = split_cols[1] if split_cols.shape[1] > 1 else np.nan
    df_melted['구분'] = df_melted['구분'].fillna('수량')

    # 10. 연도, 월 추출
    df_melted['연도'] = df_melted['진료년월_raw'].astype(str).str.extract(r'(\d{4})년')
    df_melted['월'] = df_melted['진료년월_raw'].astype(str).str.extract(r'(\d{1,2})월')
    df_melted = df_melted.dropna(subset=['월'])
    df_melted['월'] = df_melted['월'].astype(int)
    df_melted['연도'] = df_melted['연도'].astype(int)

    # ★★★ 수정: 진료년월을 'YYYY-MM' 형식으로 생성 ★★★
    df_melted['진료년월'] = (
        df_melted['연도'].astype(str).str.zfill(4) + '-' + 
        df_melted['월'].astype(str).str.zfill(2)
    )

    # '값' 컬럼을 숫자형으로 변환 ('-' 등은 NaN으로)
    df_melted['값'] = pd.to_numeric(df_melted['값'], errors='coerce')

    # 11. Pivot
    df_pivot = df_melted.pivot_table(
        index=['약효분류코드', '약효분류명', '진료년월', '연도', '월'],
        columns='구분',
        values='값',
        aggfunc='sum'
    ).reset_index()

    # 12. 컬럼명 변경
    df_pivot.rename(columns={
        '수량': '수량_천개', 
        '사용금액': '사용금액_백만원'
    }, inplace=True)

    if '수량_천개' not in df_pivot.columns: df_pivot['수량_천개'] = np.nan
    if '사용금액_백만원' not in df_pivot.columns: df_pivot['사용금액_백만원'] = np.nan

    final_columns = ['약효분류코드', '약효분류명', '진료년월', '연도', '월', '수량_천개', '사용금액_백만원']
    df_final = df_pivot[final_columns]

    # 13. 타입 정리 및 정렬
    df_final['약효분류코드'] = df_final['약효분류코드'].astype(str)
    df_final = df_final.sort_values(by=['약효분류코드', '연도', '월']).reset_index(drop=True)

    # 14. CSV 저장
    df_final.to_csv(output_file, index=False, encoding='utf-8-sig')
    print(f"저장 완료: {output_file}")
    print(f"  → 총 {len(df_final)}행 저장됨")
    print(f"  → 진료년월 예시: {df_final['진료년월'].unique()[:5].tolist()}\n")

# --- 실행 ---

if __name__ == "__main__":
    file1 = "/workspaces/healthcareDA/data/201401_201612.xlsx"
    file2 = "/workspaces/healthcareDA/data/201701_201912.xlsx"
    
    output1 = "/workspaces/healthcareDA/data/201401_201612_transformed.csv"
    output2 = "/workspaces/healthcareDA/data/201701_201912_transformed.csv"
    
    transform_excel_to_csv(file1, output1)
    transform_excel_to_csv(file2, output2)

Processing: /workspaces/healthcareDA/data/201401_201612.xlsx
  → 헤더 행 위치: 3번째 행
저장 완료: /workspaces/healthcareDA/data/201401_201612_transformed.csv
  → 총 4500행 저장됨
  → 진료년월 예시: ['2014-01', '2014-02', '2014-03', '2014-04', '2014-05']

Processing: /workspaces/healthcareDA/data/201701_201912.xlsx
  → 헤더 행 위치: 3번째 행
저장 완료: /workspaces/healthcareDA/data/201701_201912_transformed.csv
  → 총 4464행 저장됨
  → 진료년월 예시: ['2017-01', '2017-02', '2017-03', '2017-04', '2017-05']

Processing: /workspaces/healthcareDA/data/201401_201612.xlsx
  → 헤더 행 위치: 3번째 행
저장 완료: /workspaces/healthcareDA/data/201401_201612_transformed.csv
  → 총 4500행 저장됨
  → 진료년월 예시: ['2014-01', '2014-02', '2014-03', '2014-04', '2014-05']

Processing: /workspaces/healthcareDA/data/201701_201912.xlsx
  → 헤더 행 위치: 3번째 행
저장 완료: /workspaces/healthcareDA/data/201701_201912_transformed.csv
  → 총 4464행 저장됨
  → 진료년월 예시: ['2017-01', '2017-02', '2017-03', '2017-04', '2017-05']



In [22]:
file1 = "/workspaces/healthcareDA/data/201401_201612_transformed.csv"
file2 = "/workspaces/healthcareDA/data/201701_201912_transformed.csv"
file3 = "/workspaces/healthcareDA/data/meft_2020-2025.csv"

df1 = pd.read_csv(file1)
df2 = pd.read_csv(file2)
df3 = pd.read_csv(file3)
df3 = df3.iloc[:,1:]

df = pd.concat([df1,df2,df3],  axis=0)
df.to_csv('meft_2014-2025.csv')
for idf in [df1,df2,df3]:
    display(idf.head())
    print(idf.dtypes)

,약효분류코드,약효분류명,진료년월,연도,월,수량_천개,사용금액_백만원
0,111.0,전신마취제,2014-01,2014,1,6176.0,2754.0
1,111.0,전신마취제,2014-02,2014,2,5747.0,2567.0
2,111.0,전신마취제,2014-03,2014,3,5813.0,2579.0
3,111.0,전신마취제,2014-04,2014,4,5587.0,2466.0
4,111.0,전신마취제,2014-05,2014,5,5370.0,2377.0


약효분류코드      float64
약효분류명           str
진료년월            str
연도            int64
월             int64
수량_천개       float64
사용금액_백만원    float64
dtype: object


,약효분류코드,약효분류명,진료년월,연도,월,수량_천개,사용금액_백만원
0,111.0,전신마취제,2017-01,2017,1,946.0,3210.0
1,111.0,전신마취제,2017-02,2017,2,870.0,2888.0
2,111.0,전신마취제,2017-03,2017,3,933.0,3086.0
3,111.0,전신마취제,2017-04,2017,4,843.0,2842.0
4,111.0,전신마취제,2017-05,2017,5,842.0,2999.0


약효분류코드      float64
약효분류명           str
진료년월            str
연도            int64
월             int64
수량_천개       float64
사용금액_백만원    float64
dtype: object


,약효분류코드,약효분류명,진료년월,연도,월,수량_천개,사용금액_백만원
0,110,중추신경계용약,2020-01,2020,1,NaN,NaN
1,111,전신마취제,2020-01,2020,1,1017.0,3849.0
2,112,최면진정제,2020-01,2020,1,23221.0,3355.0
3,113,항전간제,2020-01,2020,1,73785.0,19473.0
4,114,해열·진통·소염제,2020-01,2020,1,478458.0,74462.0


약효분류코드        int64
약효분류명           str
진료년월            str
연도            int64
월             int64
수량_천개       float64
사용금액_백만원    float64
dtype: object
